# Practice Lab: Code a Transformer (Lesson 4.2)

Module 4 · 6 exercises · ~120-180 min
**Run Setup cell first**


# Lesson 4.2: Code a Transformer from Scratch

6 exercises: 1E + 2M + 3H
**Run Setup cell first** (loads all Transformer classes)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np
import time

# ═══ SETUP: All Transformer classes from Lesson 4.2 ═══

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1))
    scores = scores / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, V), weights

class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.W_q(query).view(B, -1, self.num_heads,
            self.d_k).transpose(1, 2)
        K = self.W_k(key).view(B, -1, self.num_heads,
            self.d_k).transpose(1, 2)
        V = self.W_v(value).view(B, -1, self.num_heads,
            self.d_k).transpose(1, 2)
        out, _ = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(
            B, -1, self.d_model)
        return self.dropout(self.W_o(out))

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class ResidualConnection(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

class EncoderBlock(nn.Module):
    def __init__(self, d, h, ff, drop=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d, h, drop)
        self.ff = FeedForward(d, ff, drop)
        self.r1 = ResidualConnection(d, drop)
        self.r2 = ResidualConnection(d, drop)
    def forward(self, x, mask=None):
        x = self.r1(x, lambda x: self.attn(x, x, x, mask))
        x = self.r2(x, self.ff)
        return x

class Encoder(nn.Module):
    def __init__(self, d, h, ff, nl, drop=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderBlock(d, h, ff, drop) for _ in range(nl)])
        self.norm = nn.LayerNorm(d)
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class DecoderBlock(nn.Module):
    def __init__(self, d, h, ff, drop=0.1):
        super().__init__()
        self.m_attn = MultiHeadAttention(d, h, drop)
        self.c_attn = MultiHeadAttention(d, h, drop)
        self.ff = FeedForward(d, ff, drop)
        self.r1 = ResidualConnection(d, drop)
        self.r2 = ResidualConnection(d, drop)
        self.r3 = ResidualConnection(d, drop)
    def forward(self, x, enc, s_mask=None, t_mask=None):
        x = self.r1(x, lambda x: self.m_attn(x, x, x, t_mask))
        x = self.r2(x, lambda x: self.c_attn(x, enc, enc, s_mask))
        x = self.r3(x, self.ff)
        return x

class Decoder(nn.Module):
    def __init__(self, d, h, ff, nl, drop=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderBlock(d, h, ff, drop) for _ in range(nl)])
        self.norm = nn.LayerNorm(d)
    def forward(self, x, enc, s_mask=None, t_mask=None):
        for layer in self.layers:
            x = layer(x, enc, s_mask, t_mask)
        return self.norm(x)

class Transformer(nn.Module):
    def __init__(self, src_vs, tgt_vs, d=512, h=8,
                 nl=6, ff=2048, max_len=5000, dropout=0.1):
        super().__init__()
        self.src_emb = InputEmbedding(src_vs, d)
        self.tgt_emb = InputEmbedding(tgt_vs, d)
        self.pe = PositionalEncoding(d, max_len, dropout)
        self.encoder = Encoder(d, h, ff, nl, dropout)
        self.decoder = Decoder(d, h, ff, nl, dropout)
        self.proj = nn.Linear(d, tgt_vs)
    def forward(self, src, tgt, s_mask=None, t_mask=None):
        enc = self.encoder(self.pe(self.src_emb(src)), s_mask)
        dec = self.decoder(
            self.pe(self.tgt_emb(tgt)), enc, s_mask, t_mask)
        return self.proj(dec)

print("\u2705 Setup complete: all Transformer classes loaded")
print(f"  Test: {Transformer(100, 100, 64, 2, 1, 64)}")


---
## Exercise 1 (Easy): Visualize Attention Weights


In [ ]:
# YOUR CODE
words = ["The", "cat", "sat", "on", "the", "mat"]
torch.manual_seed(42)
emb = torch.randn(1, 6, 64)
out, weights = scaled_dot_product_attention(emb, emb, emb)
print(f"Shape: {weights.shape}")
print(f"Row sums: {weights.squeeze().sum(-1).tolist()}")
# TODO: plot heatmap, find top pairs


<details><summary>Solution</summary>


In [ ]:
words = ["The", "cat", "sat", "on", "the", "mat"]
torch.manual_seed(42)
emb = torch.randn(1, 6, 64)
out, weights = scaled_dot_product_attention(emb, emb, emb)
w = weights.squeeze().detach().numpy()

print(f"Shape: {weights.shape}")
sums = weights.squeeze().sum(-1)
print(f"Row sums: {[f'{s:.4f}' for s in sums.tolist()]}")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(w, cmap='Purples')
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xticklabels(words, rotation=45)
ax.set_yticklabels(words)
ax.set_xlabel('Attends TO'); ax.set_ylabel('From word')
ax.set_title('Self-Attention Weights')
for i in range(6):
    for j in range(6):
        c = 'white' if w[i,j] > 0.25 else 'black'
        ax.text(j, i, f'{w[i,j]:.2f}',
                ha='center', va='center', fontsize=8, color=c)
plt.colorbar(im); plt.tight_layout()
plt.savefig('attention_heatmap.png', dpi=150)
plt.show()

flat = [(w[i,j], words[i], words[j])
        for i in range(6) for j in range(6) if i != j]
flat.sort(reverse=True)
print("\nTop-5 attention pairs:")
for val, src, tgt in flat[:5]:
    print(f'  "{src}" -> "{tgt}": {val:.3f}')


</details>


---
## Exercise 5 (Hard): Parameter Counter


In [ ]:
# YOUR CODE
def count_params(d, nh, nl, vocab, nkv=None):
    nkv = nkv or nh
    hd = d // nh
    ffn_h = int(d * 8 / 3)
    attn = d * (nh*hd + nkv*hd + nkv*hd + nh*hd)
    ffn = d * ffn_h * 3
    norm = d * 2
    per_layer = attn + ffn + norm
    emb = vocab * d
    total = nl * per_layer + emb * 2 + d
    return total, per_layer

models = [
    ("GPT-2",       768,  12, 12, 50257,  12),
    ("LLaMA 3 8B",  4096, 32, 32, 128256, 8),
    ("LLaMA 3 70B", 8192, 64, 80, 128256, 8),
]
# TODO: print table + KV-cache + GPU count


<details><summary>Solution</summary>


In [ ]:
def count_params(d, nh, nl, vocab, nkv=None):
    nkv = nkv or nh
    hd = d // nh
    ffn_h = int(d * 8 / 3)
    attn = d * (nh*hd + nkv*hd + nkv*hd + nh*hd)
    ffn = d * ffn_h * 3
    norm = d * 2
    per_layer = attn + ffn + norm
    emb = vocab * d
    total = nl * per_layer + emb * 2 + d
    return total, per_layer

models = [
    ("GPT-2",       768,  12, 12, 50257,  12),
    ("LLaMA 3 8B",  4096, 32, 32, 128256, 8),
    ("LLaMA 3 70B", 8192, 64, 80, 128256, 8),
]

print(f"{'Model':<16} {'Params':>10}"
      f" {'FP16':>7} {'INT4':>7} {'GPUs':>5}")
print("-" * 50)
for name, d, nh, nl, vs, nkv in models:
    total, _ = count_params(d, nh, nl, vs, nkv)
    fp16 = total * 2 / 1e9
    int4 = total * 0.5 / 1e9
    gpus = max(1, int(fp16 / 70) + 1)
    print(f"  {name:<14} {total/1e9:>8.2f}B"
          f" {fp16:>5.1f}GB {int4:>5.1f}GB"
          f" {gpus:>3}x A100")

print("\nKV-cache (LLaMA 3 8B, batch=1):")
for ctx in [4096, 32768]:
    hd = 4096 // 32
    kv = 2 * 32 * 8 * hd * ctx * 2
    print(f"  {ctx:>5} ctx: {kv/1e9:.2f} GB")


</details>


---
## Exercise 4 (Hard): KV-Cache Computation Count


In [ ]:
# YOUR CODE - count attention ops with vs without cache
# TODO: implement count_ops(seq_len, prompt_len)


<details><summary>Solution</summary>


In [ ]:
def count_ops(seq_len, prompt_len=10):
    # Naive: each step recomputes ALL attention
    naive = sum(i * i for i in range(1, seq_len + 1))
    # Cached: prefill once, then 1 new Q per step
    cached = (prompt_len * prompt_len
              + sum(prompt_len + i
                    for i in range(1, seq_len - prompt_len + 1)))
    return naive, cached

print(f"{'Tokens':>7} {'Naive':>14} {'Cached':>12}"
      f" {'Speedup':>9}")
print("-" * 48)
for n in [50, 100, 1000, 4096]:
    naive, cached = count_ops(n)
    print(f"  {n:>5}  {naive:>12,}  {cached:>10,}"
          f"  {naive/cached:>7.1f}x")


</details>


---
## Exercise 2 (Medium): Change the Architecture
Full starter code, hints, expected output in practice-lab-4_2.html. Needs Colab for training.


In [ ]:
# Exercise 2: Change the Architecture
# See practice-lab-4_2.html for details.
# Needs PyTorch training (run in Colab).
pass


---
## Exercise 3 (Medium): RMSNorm + SwiGLU Swap
Full starter code, hints, expected output in practice-lab-4_2.html. Needs Colab for training.


In [ ]:
# Exercise 3: RMSNorm + SwiGLU Swap
# See practice-lab-4_2.html for details.
# Needs PyTorch training (run in Colab).
pass


---
## Exercise 6 (Challenge): Pig Latin Translation
Full starter code, hints, expected output in practice-lab-4_2.html. Needs Colab for training.


In [ ]:
# Exercise 6: Pig Latin Translation
# See practice-lab-4_2.html for details.
# Needs PyTorch training (run in Colab).
pass
